# Week 1 Concepts — 컴퓨터 비전 & 시계열

1~4일차를 위한 실행 가능한 동반 노트북입니다. 각 섹션은 대응하는 일별 노트
([Day1](../daily/Day1_cnn-vs-vit.ko.md), [Day2](../daily/Day2_diffusion-models.ko.md),
[Day3](../daily/Day3_time-series-fundamentals.ko.md), [Day4](../daily/Day4_forecasting-methods.ko.md))
로 이어지며, 같은 예제를 재사용하므로 여기서 출력되는 shape와 숫자는 그 노트에서
언급하는 값과 동일합니다.

**실행에 대하여**: 아래 numpy/pandas/statsmodels 셀들(1일차의 패치 임베딩·어텐션
shape 추적, 2일차의 forward diffusion 노이즈 스케줄, 3일차 전체, 4일차 전체)은
GPU 없는 순수 numpy/pandas/statsmodels 환경에서 실제로 실행했으며, 각 셀 아래
표시된 출력은 그 실행에서 나온 실제 출력을 그대로 옮긴 것입니다. 두 개의
`transformers`/`diffusers` 파이프라인 셀(1일차 끝, 2일차 끝)은 해당 라이브러리의
실제 최신 API를 사용하지만, 이 환경에는 GPU도 torch/transformers/diffusers
설치도 없어 여기서는 **실행하지 않았습니다** — 실행이 검증된 스크립트가 아니라
기술적으로 정확한 참고 코드로 포함했습니다.


## Day 1: CNN vs. 비전 트랜스포머

먼저, conv 레이어 하나와 ViT의 패치 임베딩 -> CLS 토큰 -> 위치 임베딩 -> 셀프
어텐션 파이프라인 전체를 numpy만으로 처음부터 구현해 shape를 추적한다. 텐서가
매 단계에서 어떻게 변하는지 확인하는 데 딥러닝 프레임워크는 전혀 필요 없다.
그다음, 신뢰도 기반 검토 플래그를 포함한 배치 이미지 분류를 위한 실제(단, 여기서는
실행하지 않은) `transformers` 파이프라인 패턴을 소개한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# ---- 이미지를 텐서로 ----
B, C, H, W = 1, 3, 224, 224
image = rng.uniform(0, 1, size=(B, C, H, W)).astype(np.float32)
# shape: (batch=1, channels=3, H=224, W=224)
print("image:", image.shape, image.dtype)

# ---- CNN: conv 레이어 하나를 직접 구현 (im2col 방식) --
# 라이브러리 호출 뒤에 숨기지 않고 shape 계산이 그대로 드러나도록 한다
kernel = rng.normal(size=(8, C, 3, 3)).astype(np.float32)
# shape: (out_channels=8, in_channels=3, kh=3, kw=3)

def conv2d_naive(x, w, stride=1):
    B, C, H, W = x.shape
    OC, _, kh, kw = w.shape
    oh = (H - kh) // stride + 1          # 패딩이 없으므로 출력 높이가 줄어든다
    ow = (W - kw) // stride + 1          # 출력 너비도 마찬가지로 줄어든다
    out = np.zeros((B, OC, oh, ow), dtype=np.float32)
    w_flat = w.reshape(OC, -1)            # (OC, C*kh*kw) -- 필터마다 펼치기
    for i in range(oh):
        for j in range(ow):
            # 이 출력 위치가 의존하는 수용 영역 패치를 가져온다
            patch = x[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]   # (B, C, kh, kw)
            patch_flat = patch.reshape(B, -1)                             # (B, C*kh*kw)
            out[:, :, i, j] = patch_flat @ w_flat.T                       # (B, OC) 필터별 내적
    return out

# 16x16 크롭에서만 실행 -- 224x224 전체에 순수 파이썬 루프를 돌리면
# shape 데모용으로는 불필요하게 느리다
small_crop = image[:, :, :16, :16]                  # shape: (1, 3, 16, 16)
feature_map = conv2d_naive(small_crop, kernel)      # shape: (1, 8, 14, 14)
print("conv feature_map:", feature_map.shape)   # 공간 축마다 (16-3)/1+1 = 14

# ---- ViT: 패치화 + 선형 투영 + CLS 토큰 + 위치 임베딩 ----
patch_size = 16
n_patches_h = H // patch_size                       # 224 / 16 = 14
n_patches_w = W // patch_size                        # 14
n_patches = n_patches_h * n_patches_w                 # 196
patch_dim = C * patch_size * patch_size               # 3*16*16 = 768
embed_dim = 768                                        # ViT-Base 은닉 차원

# 이미지를 겹치지 않는 패치 그리드로 재구성한 뒤, 각 패치를 벡터 하나로
# 펼친다 -- 아직 학습된 가중치는 전혀 관여하지 않는다
x = image.reshape(B, C, n_patches_h, patch_size, n_patches_w, patch_size)
x = x.transpose(0, 2, 4, 1, 3, 5)      # 각 패치의 픽셀들을 하나로 묶는다
patches = x.reshape(B, n_patches, patch_dim)
print("patches:", patches.shape)  # (1, 196, 768) -- 패치 196개, 아직 원시 768차원 픽셀 벡터

W_proj = rng.normal(scale=0.02, size=(patch_dim, embed_dim)).astype(np.float32)
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    patch_tokens = patches @ W_proj
print("patch_tokens:", patch_tokens.shape)  # (1, 196, 768) -- 이제는 학습된 임베딩

cls_token = rng.normal(scale=0.02, size=(1, 1, embed_dim)).astype(np.float32)
cls_tokens = np.repeat(cls_token, B, axis=0)         # shape: (1, 1, 768)
tokens = np.concatenate([cls_tokens, patch_tokens], axis=1)
print("tokens w/ CLS:", tokens.shape)  # (1, 197, 768) -- CLS 토큰이 이제 0번 위치

pos_embed = rng.normal(scale=0.02, size=(1, n_patches + 1, embed_dim)).astype(np.float32)
tokens = tokens + pos_embed
print("tokens + pos_embed:", tokens.shape)  # (1, 197, 768) -- shape 동일, 값에 위치 정보 인코딩

# ---- 197개 토큰 전체 시퀀스에 대한 셀프 어텐션 레이어 하나 ----
Wq = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)
Wk = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)
Wv = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)

with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    Q = tokens @ Wq   # shape: (1, 197, 768) -- 토큰마다 쿼리 벡터 하나
    K = tokens @ Wk   # shape: (1, 197, 768) -- 토큰마다 키 벡터 하나
    V = tokens @ Wv   # shape: (1, 197, 768) -- 토큰마다 값 벡터 하나

    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(embed_dim)
    # shape: (1, 197, 197) -- scores[0, i, j] = 토큰 i가 토큰 j에 얼마나 주의를 기울이는가
    print("attention scores:", scores.shape)

    scores = scores - scores.max(axis=-1, keepdims=True)   # 수치 안정성
    attn = np.exp(scores)
    attn = attn / attn.sum(axis=-1, keepdims=True)          # 각 행의 합은 1
    out = attn @ V
print("attention output:", out.shape)  # (1, 197, 768) -- 모든 토큰의 값 벡터를 가중 혼합

print("row sums ~1:", round(float(attn[0, 0].sum()), 6))
print("CLS -> last patch weight:", float(attn[0, 0, -1]))
print("CLS -> first patch weight:", float(attn[0, 0, 1]))


image: (1, 3, 224, 224) float32
conv feature_map: (1, 8, 14, 14)
patches: (1, 196, 768)
patch_tokens: (1, 196, 768)
tokens w/ CLS: (1, 197, 768)
tokens + pos_embed: (1, 197, 768)
attention scores: (1, 197, 197)
attention output: (1, 197, 768)
row sums ~1: 1.0
CLS -> last patch weight: 0.005082810592993156
CLS -> first patch weight: 0.005071407237220381


### 참고 패턴: `transformers`를 이용한 배치 분류 (여기서는 실행하지 않음)

실제 최신 `transformers` API다. `torch`와 다운로드된 체크포인트가 필요하므로 이
노트북에서는 실행하지 않는다 -- 하지만 텐서 shape는 위 ViT 설명에서 그대로
이어진다. `pixel_values`는 `(1, 3, 224, 224)`이고, `logits`는 체크포인트가 쓰는
분류 헤드에 따라 `(1, num_labels)`이다.


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch
import pandas as pd

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")
model.eval()  # dropout 등을 비활성화 -- 이건 추론이지 학습이 아니다

camera_trap_photos = ["trailcam_0091.jpg", "trailcam_0092.jpg", "trailcam_0093.jpg"]

rows = []
for path in camera_trap_photos:
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    # inputs["pixel_values"] shape: (1, 3, 224, 224)

    with torch.no_grad():  # 추론에는 역전파가 필요 없다
        logits = model(**inputs).logits
    # logits shape: 이 ImageNet-1k 체크포인트 기준 (1, 1000)

    probs = logits.softmax(dim=-1)[0]        # shape: (1000,), 합이 1.0
    top_prob, top_idx = probs.max(dim=-1)

    rows.append({
        "file": path,
        "predicted_label": model.config.id2label[top_idx.item()],
        "confidence": round(top_prob.item(), 3),
        "needs_review": top_prob.item() < 0.6,
    })

results = pd.DataFrame(rows)  # -> DataFrame, columns [file, predicted_label, confidence, needs_review], len=3
results


## Day 2: 디퓨전 모델과 텍스트-이미지 생성

먼저, forward diffusion 노이즈 스케줄(타임스텝에 따라 신호 대 노이즈 비율이 어떻게
감쇠하는지)과, U-Net의 공간적 위치가 텍스트 토큰 임베딩에 주의를 기울이게 하는
크로스 어텐션의 shape 산술을 numpy로 처음부터 추적한다. 그다음, `num_inference_steps`와
`guidance_scale` 설정을 비교하는 실제(단, 여기서는 실행하지 않은) `diffusers`
파이프라인 패턴을 소개한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# ---- VAE로 인코딩된 이미지를 대신하는 작은 "잠재변수" ----
# 실제 Stable Diffusion 잠재변수는 512x512 이미지에 대해 (batch, 4, 64, 64)이며,
# 산술이 대표성을 갖도록 여기서도 같은 shape를 사용한다.
B, C, Hl, Wl = 1, 4, 64, 64
x0 = rng.uniform(-1, 1, size=(B, C, Hl, Wl)).astype(np.float32)
print("x0 (clean latent):", x0.shape, "mean", round(float(x0.mean()), 4), "std", round(float(x0.std()), 4))

# ---- 고정된 선형 노이즈 스케줄 (학습 없음) ----
T = 1000
betas = np.linspace(1e-4, 0.02, T).astype(np.float32)   # shape: (1000,)
alphas = 1.0 - betas                                      # shape: (1000,)
alpha_bars = np.cumprod(alphas)                            # shape: (1000,), 단조 감소

def forward_diffuse(x0, t_index, alpha_bars, rng):
    """x0로부터 x_t를 닫힌 형태로 직접 샘플링, 중간 단계를 모두 건너뛴다."""
    a_bar = alpha_bars[t_index]
    noise = rng.normal(size=x0.shape).astype(np.float32)   # x0와 shape 동일
    xt = np.sqrt(a_bar) * x0 + np.sqrt(1 - a_bar) * noise    # x0와 shape 동일
    return xt, noise

print("\nt      xt.shape          signal_scale  noise_scale")
for t_index in [0, 249, 499, 999]:
    xt, noise = forward_diffuse(x0, t_index, alpha_bars, rng)
    signal_scale = float(np.sqrt(alpha_bars[t_index]))
    noise_scale = float(np.sqrt(1 - alpha_bars[t_index]))
    print(f"{t_index:4d}   {str(xt.shape):16s}  {signal_scale:.4f}        {noise_scale:.4f}")

# ---- 크로스 어텐션 shape 추적: 잠재변수의 공간 토큰이 텍스트 토큰에 주의를 기울인다 ----
n_spatial_tokens = Hl * Wl                      # 64*64 = 4096
latent_seq = x0.reshape(B, C, n_spatial_tokens).transpose(0, 2, 1)  # (B, 4096, 4)
print("\nlatent flattened to sequence:", latent_seq.shape)

n_text_tokens, text_dim = 77, 768  # CLIP의 고정 컨텍스트 길이 x 임베딩 차원
text_embeddings = rng.normal(scale=0.02, size=(B, n_text_tokens, text_dim)).astype(np.float32)
print("text_embeddings:", text_embeddings.shape)

attn_dim = 128
Wq = rng.normal(scale=0.02, size=(C, attn_dim)).astype(np.float32)
Wk = rng.normal(scale=0.02, size=(text_dim, attn_dim)).astype(np.float32)
Wv = rng.normal(scale=0.02, size=(text_dim, attn_dim)).astype(np.float32)

with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    Q = latent_seq @ Wq          # (B, 4096, attn_dim) -- 공간 위치마다 쿼리 하나
    K = text_embeddings @ Wk     # (B, 77, attn_dim)   -- 텍스트 토큰마다 키 하나
    V = text_embeddings @ Wv     # (B, 77, attn_dim)
    print("Q (spatial queries):", Q.shape)
    print("K, V (text keys/values):", K.shape, V.shape)

    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(attn_dim)  # (B, 4096, 77)
    print("cross-attention scores:", scores.shape)  # 잠재변수의 픽셀 4096개가 각각 텍스트 토큰 77개를 채점

    scores = scores - scores.max(axis=-1, keepdims=True)
    attn = np.exp(scores)
    attn = attn / attn.sum(axis=-1, keepdims=True)
    cross_out = attn @ V  # (B, 4096, attn_dim)
print("cross-attention output:", cross_out.shape)  # 픽셀별 텍스트 조건부 특징


x0 (clean latent): (1, 4, 64, 64) mean 0.0034 std 0.5764

t      xt.shape          signal_scale  noise_scale
   0   (1, 4, 64, 64)    0.9999        0.0100
 249   (1, 4, 64, 64)    0.7239        0.6899
 499   (1, 4, 64, 64)    0.2803        0.9599
 999   (1, 4, 64, 64)    0.0064        1.0000

latent flattened to sequence: (1, 4096, 4)
text_embeddings: (1, 77, 768)
Q (spatial queries): (1, 4096, 128)
K, V (text keys/values): (1, 77, 128) (1, 77, 128)
cross-attention scores: (1, 4096, 77)
cross-attention output: (1, 4096, 128)


### 참고 패턴: `diffusers`를 이용한 배치 생성 (여기서는 실행하지 않음)

실제 최신 `diffusers` API다. `torch`와 `diffusers`가 필요하고 GPU를 강력히
권장하므로 이 노트북에서는 실행하지 않는다 -- 하지만 루프 안의 모든 shape는
2일차 노트에서 다룬 파이프라인 메커니즘 그대로다. `(1, 4, 64, 64)` 잠재변수가
`num_inference_steps`번의 U-Net 호출에 걸쳐 노이즈가 제거된 뒤, VAE가 이를
`(1, 3, 512, 512)`에 해당하는 출력 이미지로 디코딩한다.


In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import pandas as pd

pipe = StableDiffusionPipeline.from_pretrained(
    "segmind/small-sd",          # 증류된 경량 체크포인트
    torch_dtype=torch.float32,   # CPU라면 float32; GPU에서는 보통 float16
)

sticker_prompts = [
    "a cartoon sticker of a sleepy cat wearing headphones, flat vector style",
    "a cartoon sticker of a rocket ship with a smiling face, flat vector style",
]

rows = []
for prompt in sticker_prompts:
    for steps, guidance in [(15, 4.0), (30, 7.5), (30, 12.0)]:
        # 호출마다 전체 역방향 디퓨전 루프가 실행된다: `steps`번의 순차적
        # U-Net 평가로 64x64x4 잠재변수의 노이즈를 제거한 뒤 VAE 디코드 한 번
        image = pipe(prompt, num_inference_steps=steps, guidance_scale=guidance).images[0]
        fname = f"sticker_{sticker_prompts.index(prompt)}_{steps}_{guidance}.png"
        image.save(fname)
        rows.append({"prompt": prompt, "steps": steps, "guidance_scale": guidance, "file": fname})

comparison = pd.DataFrame(rows)  # -> DataFrame, len = 프롬프트 2개 * 설정 3개 = 6
comparison


## Day 3: 시계열 기초

합성 2년치 일별 "유동인구" 시리즈(트렌드 + 주간 계절성 + 노이즈) -- Day 3와 Day 4
노트 전체에서 사용하는 것과 같은 시리즈다. 아래에서: 원본 시리즈 대 트렌드 제거된
시리즈의 자기상관(강한 트렌드가 모든 시차를 어떻게 부풀리고 실제 계절 피크를
가리는지 확인), 리샘플링, `seasonal_decompose` 전체 호출, 그리고 원본 대
계절성 제거 데이터에 대한 이상치 탐지로 계절 피크 거짓 양성 문제를 직접
확인한다 -- 진짜 이상치 하나를 주입해서 그것만 플래그되는지 확인하는 것도 포함된다.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf

rng = np.random.default_rng(7)

# ---- 합성 일별 유동인구 시리즈: 트렌드 + 주간 계절성 + 노이즈 ----
dates = pd.date_range("2023-01-01", periods=730, freq="D")    # 2년치 일별 데이터
trend = np.linspace(80, 160, len(dates))                       # 완만한 선형 성장
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)           # 주기-7 계절 파동
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
print("foot_traffic:", foot_traffic.shape, foot_traffic.dtype)  # -> Series, len=730, float64

# ---- 원본 시리즈에 대한 자기상관: 트렌드가 모든 시차를 부풀린다 ----
acf_raw = acf(foot_traffic, nlags=14)
print("\nACF on raw series (트렌드가 모든 시차를 부풀린다):")
for lag in [0, 1, 6, 7, 8, 14]:
    marker = "  <-- local peak" if lag in (7, 14) else ""
    print(f"  lag {lag:2d}: {acf_raw[lag]:+.3f}{marker}")

# ---- 트렌드 제거된 시리즈에 대한 자기상관: 계절 피크가 명확해진다 ----
trend_est = seasonal_decompose(foot_traffic, model="additive", period=7).trend
detrended = (foot_traffic - trend_est).dropna()
acf_detrended = acf(detrended, nlags=14)
print("\nACF on detrended series (이제 시차 7이 명확한 피크):")
for lag in [0, 1, 3, 7, 14]:
    marker = "  <-- clear peak" if lag == 7 else ""
    print(f"  lag {lag:2d}: {acf_detrended[lag]:+.3f}{marker}")

# ---- 리샘플링: 일별 -> 주별 다운샘플링 ----
weekly_avg = foot_traffic.resample("W").mean()
print("\nweekly_avg:", weekly_avg.shape)  # 일별 len=730 -> 주별 len ~104-106

# ---- 전체 분해 ----
result = seasonal_decompose(foot_traffic, model="additive", period=7)
print("\nresult.trend:", result.trend.shape, result.trend.dtype)
print("result.seasonal:", result.seasonal.shape)
print("result.resid:", result.resid.shape)
print("NaNs in trend (중심 이동창의 경계 효과):", int(result.trend.isna().sum()))
print("seasonal component, first 14 values (7마다 반복):")
print(result.seasonal.head(14).round(2).tolist())


foot_traffic: (730,) float64

ACF on raw series (트렌드가 모든 시차를 부풀린다):
  lag  0: +1.000
  lag  1: +0.913
  lag  6: +0.897
  lag  7: +0.953  <-- local peak
  lag  8: +0.888
  lag 14: +0.930  <-- local peak

ACF on detrended series (이제 시차 7이 명확한 피크):
  lag  0: +1.000
  lag  1: +0.544
  lag  3: -0.833
  lag  7: +0.893  <-- clear peak
  lag 14: +0.885

weekly_avg: (106,)

result.trend: (730,) float64
result.seasonal: (730,)
result.resid: (730,)
NaNs in trend (중심 이동창의 경계 효과): 6
seasonal component, first 14 values (7마다 반복):
[-11.4, 0.1, 11.28, 14.41, 6.87, -6.81, -14.44, -11.4, 0.1, 11.28, 14.41, 6.87, -6.81, -14.44]


이제 이상치 탐지: 원본 시리즈 대 계절성 제거된 잔차에 대한 단순 이동 3-시그마
탐지기를 비교하고, 진짜 이상치 하나를 주입해서 잔차 기반 버전이 실제로 그것만
잡아내는지 확인한다.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
result = seasonal_decompose(foot_traffic, model="additive", period=7)

# ---- 원본 시리즈에 대한 이상치 검사: 이동 윈도우가 주중/주말 기준선을 섞어버린다 ----
roll_mean_raw = foot_traffic.rolling(14).mean()
roll_std_raw = foot_traffic.rolling(14).std()
raw_anomaly = (foot_traffic - roll_mean_raw).abs() > 3 * roll_std_raw
print("Raw-series anomalies flagged (rolling 14d, 3-sigma):", int(raw_anomaly.sum()))

# ---- 계절성이 제거된 잔차에 대한 이상치 검사 ----
resid = result.resid.dropna()
roll_mean_resid = resid.rolling(14).mean()
roll_std_resid = resid.rolling(14).std()
resid_anomaly = (resid - roll_mean_resid).abs() > 3 * roll_std_resid
print("Deseasonalized-residual anomalies flagged:", int(resid_anomaly.sum()))

# ---- 진짜 이상치 하나를 주입하고, 그것만 플래그되는지 확인 ----
foot_traffic_with_spike = foot_traffic.copy()
foot_traffic_with_spike.iloc[400] += 60   # 예: 지역 행사, 화제가 된 게시물
result2 = seasonal_decompose(foot_traffic_with_spike, model="additive", period=7)
resid2 = result2.resid.dropna()
roll_mean2 = resid2.rolling(14).mean()
roll_std2 = resid2.rolling(14).std()
anomaly2 = (resid2 - roll_mean2).abs() > 3 * roll_std2
flagged_dates = resid2.index[anomaly2]
print("\nWith an injected spike at index 400:")
print("  flagged dates:", [d.strftime('%Y-%m-%d') for d in flagged_dates])
print("  actual spike date:", foot_traffic.index[400].strftime("%Y-%m-%d"))


Raw-series anomalies flagged (rolling 14d, 3-sigma): 0
Deseasonalized-residual anomalies flagged: 0

With an injected spike at index 400:
  flagged dates: ['2024-02-05']
  actual spike date: 2024-02-05


## Day 4: 예측 기법과 평가

같은 유동인구 시리즈를 시간 기준점에서 분할한다(마지막 60일 홀드아웃). 먼저
원본 대 차분된 시리즈에 대한 증강 디키-풀러 정상성 검정을 실행한다. 그다음
예측 방법의 전체 사다리 -- 이동평균, 단순 지수평활, Holt, Holt-Winters -- 를
홀드아웃 구간에서 MAE / MAPE / RMSE로 점수 매긴다. 마지막으로, 단일 점 예측
대신 시뮬레이션된 예측 구간을 만들고, 실제 값이 그 밴드 안에 실제로 얼마나
자주 들어왔는지 확인한다.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(7)

# ---- Day 3와 동일한 합성 유동인구 시리즈 ----
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")

train, test = foot_traffic[:-60], foot_traffic[-60:]
print("train:", train.shape, "test:", test.shape)  # -> (670,) (60,)

# ---- 정상성 검사: 원본 vs. 차분된 시리즈에 대한 ADF 검정 ----
stat, p_value, *_ = adfuller(foot_traffic)
print(f"\nADF on raw series:        stat={stat:.3f}  p={p_value:.4f}  "
      f"({'stationary' if p_value < 0.05 else 'NOT stationary'})")

diffed = foot_traffic.diff().dropna()   # -> Series, len=729 (차분으로 한 지점 손실)
stat_d, p_value_d, *_ = adfuller(diffed)
print(f"ADF on differenced series: stat={stat_d:.3f}  p={p_value_d:.6f}  "
      f"({'stationary' if p_value_d < 0.05 else 'NOT stationary'})")


train: (670,) test: (60,)

ADF on raw series:        stat=-0.568  p=0.8780  (NOT stationary)
ADF on differenced series: stat=-10.298  p=0.000000  (stationary)


예측 방법의 사다리를 끝까지 피팅하고 점수 매긴다:

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
train, test = foot_traffic[:-60], foot_traffic[-60:]

# ---- 나이브에서 계절성까지, 방법의 사다리 ----
def moving_average_forecast(train, horizon, window=7):
    last_avg = train.iloc[-window:].mean()
    return pd.Series([last_avg] * horizon, index=test.index)  # 평탄선, 트렌드/계절 없음

ma_forecast = moving_average_forecast(train, len(test))

ses_model = ExponentialSmoothing(train, trend=None, seasonal=None).fit()
ses_forecast = ses_model.forecast(len(test))

holt_model = ExponentialSmoothing(train, trend="add", seasonal=None).fit()
holt_forecast = holt_model.forecast(len(test))

hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()
hw_forecast = hw_model.forecast(len(test))
print("hw_forecast:", hw_forecast.shape)  # -> Series, len=60

# ---- 홀드아웃 테스트 세트로 모든 방법에 점수 매기기 ----
def mae(y, yhat): return float(np.mean(np.abs(y - yhat)))
def mape(y, yhat): return float(np.mean(np.abs((y - yhat) / y)) * 100)
def rmse(y, yhat): return float(np.sqrt(np.mean((y - yhat) ** 2)))

print(f"\n{'Method':16s} {'MAE':>6s}  {'MAPE(%)':>7s}  {'RMSE':>6s}")
for name, fc in [("MovingAvg", ma_forecast), ("SES", ses_forecast),
                  ("Holt", holt_forecast), ("Holt-Winters", hw_forecast)]:
    print(f"{name:16s} {mae(test, fc):6.2f}  {mape(test, fc):6.2f}   {rmse(test, fc):6.2f}")


hw_forecast: (60,)

Method              MAE  MAPE(%)    RMSE
MovingAvg          9.80    6.34    11.46
SES               10.17    6.66    11.87
Holt              11.29    7.53    13.41
Holt-Winters       3.32    2.12     4.17


숫자 하나 대신 범위: 그럴듯한 경로 여러 개를 시뮬레이션해 90% 밴드를
보고한 뒤, 실제 값이 그 안에 얼마나 자주 들어왔는지 실증적으로 확인한다.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
train, test = foot_traffic[:-60], foot_traffic[-60:]

hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()

# ---- 단일 점 예측 대신 시뮬레이션을 통한 예측 구간 ----
simulations = hw_model.simulate(len(test), repetitions=200, error="add", random_state=7)
print("simulations:", simulations.shape)  # -> DataFrame, (60, 200): 60 스텝 x 시뮬레이션 경로 200개

lower = simulations.quantile(0.05, axis=1)   # Series, len=60 -- 일자별 5번째 백분위
upper = simulations.quantile(0.95, axis=1)   # Series, len=60 -- 일자별 95번째 백분위
print("lower/upper band shape:", lower.shape, upper.shape)

in_band = ((test.values >= lower.values) & (test.values <= upper.values)).mean()
print(f"fraction of true test points inside the 90% band: {in_band:.2%}")


simulations: (60, 200)
lower/upper band shape: (60,) (60,)
fraction of true test points inside the 90% band: 83.33%
